# Does concept frequency predict whether few-shot adaptation helps?

Read the markdown cells top to bottom to check what this notebook does.

Each section says **what** it does, **why**, its **source** (paper and page, or our own choice), and its **status**:
✅ checked, ⚠ open issue, 📝 our own decision.

## The question, in four steps

| | claim | status |
|---|---|---|
| 1 | Zero-shot CLIP models are not fully "zero-shot": the pathology names appear in the training reports | measured here (section 2) |
| 2 | Pathologies named **rarely** get **lower** zero-shot AUC | prior work in general images (Udandarao et al. 2024); we test it in radiology (section 7) |
| 3 | So we add K labelled images per class: few-shot adaptation (SS-Text) | method (section 8) |
| 4 | Adaptation helps the **rare** pathologies and **hurts** the common ones | **new**, our hypothesis (section 9) |

Written as two lines over f, the share of training image–text pairs whose report asserts the
pathology (section 2 defines f exactly):

$$\text{(1)}\;\; \text{AUC}_{\text{zero-shot}} = a\,\log_{10} f + b \qquad\qquad \text{(2)}\;\; \Delta\text{AUC}_{\text{few-shot}} = c\,\log_{10} f + d$$

The hypothesis is **a > 0** and **c < 0**.

* **Source of (1):** Udandarao et al. describe a "log-linear" relationship in words and plots
  only: *"a consistent linear relationship between CLIP's zero-shot performance on a concept and
  the log-scaled pretraining concept frequency"* (Fig. 2 caption, PDF p.5). **The equations are our
  formalization**, not theirs. In the paper, cite them for the relationship, not the formula.
* **"Law 1 / Law 2"** are nicknames for (1) and (2) in this notebook and its code (`laws.py`, `L1`, `L2`).
  With 14 points they are *relationships*, not laws. Use "relationship" in the paper.

## Design decisions

📝 **Single checkpoint, no ensemble.** \
CheXzero paper ensembles its top-10 checkpoints. But we use the single best, to remove one moving part.\
CheXzero Table 5 (PDF p.6) reports **0.878** for the best single model and **0.889** for the ensemble.\
⚠ The Table 5 caption says these are on the CheXpert **validation** set, while the paper's headline 0.889 is on the **test** set (abstract, PDF p.1).\
Comparing our test AUC with 0.878 is therefore approximate.

📝 **Each dataset selects its own checkpoint**, on its own selection split (section 4).\
An earlier run ranked checkpoints on CheXpert validation only and reused the choice for ChestX-ray14, which favoured CheXpert. \
The cost: the two datasets may use different checkpoints, so within-dataset results are the primary evidence.

```
CheXpert  ->  select on CheXpert val (200 studies)       ->  support from CheXpert train (10,000)
NIH       ->  select on 6,000 of the 20,000 promptdev    ->  support from the other 14,000
```

## Where things live

| what | where |
|---|---|
| every path and setting | `config/freq_v1.yaml` (`config.show(CFG)` prints it) |
| concept vocabulary (synonyms, negation) | `config/concept_synonyms.yaml` |
| all functions | `utils/freq/` (one module per stage; see its `__init__.py`) |
| outputs and caches | `runs/freq_v1/results/` |
| source passages from the papers | `files/papers/name_freq/READING_GUIDE.md` |

**Runtime:** checkpoint ranking about 55 min the first time; everything else a few minutes.
Every expensive step writes a cache and is skipped on re-run.
**Kernel:** `hj` conda environment (`/home/hj/anaconda3/envs/hj/bin/python`).

In [16]:
import os, sys, warnings
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

ROOT     = "/mnt/My_Doc/github/xray_clip/"
CHEXZERO = ROOT + "CheXzero/"
os.chdir(ROOT)                                # work from the repo root
sys.path[:0] = [ROOT, CHEXZERO]               # our utils/ and the CheXzero modules are importable

from utils import config, plot
from utils.freq import mentions, data, clip_eval, fewshot, laws
from eval_nih import NIH_LABELS, LABEL_PROMPTS, build_groundtruth as nih_groundtruth
from select_checkpoints import build_groundtruth as cxp_val_groundtruth

CFG  = config.load_experiment("config/freq_v1.yaml")      # every setting and path
config.show(CFG)                                          # everything
# config.show(CFG, ["experiment", "eval", "files"])       # or only these sections

F, E = CFG["files"], CFG["eval"]
assert not config.missing_files(CFG), config.missing_files(CFG)

EXP      = os.environ.get("F6_EXP_DIR", CFG["experiment"]["dir"])
OUT      = os.path.join(EXP, "results"); os.makedirs(OUT, exist_ok=True)
CKPT_DIR = CFG["experiment"]["ckpt_dir"]
DEVICE   = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
CXP5     = CFG["cxp_labels_5"]
NIH14    = list(NIH_LABELS)

print(f"\nconfig     : {CFG['config_file']}")
print(f"experiment : {EXP}")
print(f"checkpoints: {CKPT_DIR}")
print(f"device     : {DEVICE}")

datasets: config/datasets.yaml
experiment:
  dir: /mnt/My_Doc/github/xray_clip/runs/freq_v1
  ckpt_dir: /mnt/My_Doc/github/xray_clip/runs/chexpert_v1/checkpoints/pt-imp-cxp
train:
  batch_size: 64
  epochs: 8
  lr: 0.0001
  optimizer: sgd
  momentum: 0.9
  save_interval: 200
min_checkpoints: 200
eval:
  batch: 256
  ctx: 77
  seeds: 20
  k_sweep: [1, 2, 4, 8, 16, 32, 64]
  n_sel: 6000
  seed: 0
  k_ref: 64
  k_report: [16, 64]
  n_boot: 1000
steps_per_epoch: 5893
reference:
  chexzero_single: 0.878
synonyms: /mnt/My_Doc/github/xray_clip/config/concept_synonyms.yaml
frequency:
  primary: labeler_pos_%
  spacy_python: ~/.venvs/spacy372/bin/python
cxp_labels_5: [Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion]
cxp_labels_all: [Enlarged Cardiomediastinum, Cardiomegaly, Lung Opacity, Lung Lesion, Edema, Consolidation, Pneumonia, Atelectasis, Pneumothorax, Pleural Effusion, Pleural Other, Fracture]
chexzero_data:
  path: /mnt/My_Doc/github/xray_clip/CheXzero/data
  impressi

## 1. Train the model (reused, not retrained)

**What:** CheXzero's own training recipe, run once on MIMIC-CXR: 377,110 image–report pairs,
**impression section only**, 8 epochs, SGD lr 1e-4, momentum 0.9, batch 64, a checkpoint every 200 steps.

**Source:** CheXzero paper, *"We utilize the impressions section of each text report"* (PDF p.7).
Their released code turns a report without an impression into the literal text `NO IMPRESSION`.

**Status:**
* ✅ We **reuse** the 236 checkpoints of the earlier run in `runs/chexpert_v1/` (same recipe, see
  its `run_all.sh`). The cell below finds them and skips training. To retrain, point
  `experiment.ckpt_dir` in the YAML at an empty folder.
* 📝 8 epochs rather than the paper's 4, so checkpoint selection can choose across a wider range
  (section 4 shows where the best one falls).
* ⚠ `CheXzero/data_process.py` now contains two blocks we added later (a findings fallback and
  findings appending). `mimic_impressions.csv` was built **before** them, so this model is
  unaffected, but rerunning preprocessing would change the text. See `CheXzero/LOCAL_CHANGES.md`.

In [17]:
import subprocess, h5py
os.makedirs(CKPT_DIR, exist_ok=True)
list_ckpts = lambda: sorted(os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR) if f.endswith(".pt"))
# print(list_ckpts())
# print(CFG["min_checkpoints"], len(list_ckpts()))

if len(list_ckpts()) >= CFG["min_checkpoints"]:
    print(f"SKIP training -- {len(list_ckpts())} checkpoints already in {CKPT_DIR}")
else:
    # alignment gate: a silent mismatch here trains on wrong image/text pairs
    with h5py.File(F["mimic"]["cxr_h5"], "r") as f:
        n_img = f["cxr"].shape[0]
    txt = pd.read_csv(F["chexzero_data"]["impressions"])
    assert n_img == len(txt), f"MISALIGNED: h5 {n_img:,} vs text {len(txt):,}"
    print(f"gate OK: {n_img:,} images, NO IMPRESSION {(txt.impression.astype(str)=='NO IMPRESSION').mean():.1%}")

    args = {**CFG["train"], "cxr_filepath": F["mimic"]["cxr_h5"], "txt_filepath": F["chexzero_data"]["impressions"],
            "save_dir": os.path.dirname(CKPT_DIR), "model_name": os.path.basename(CKPT_DIR)}
    cmd  = [sys.executable, "-u", "run_train.py"] + [x for k, v in args.items() for x in (f"--{k}", str(v))]
    print("training (~1h35m) ...", " ".join(cmd[3:]))
    with open(os.path.join(EXP, "train.log"), "w") as log:
        subprocess.run(cmd, cwd=CHEXZERO, stdout=log, stderr=subprocess.STDOUT, check=True)

CKPT_PATHS = list_ckpts()
print(f"{len(CKPT_PATHS)} checkpoints available")

SKIP training -- 236 checkpoints already in /mnt/My_Doc/github/xray_clip/runs/chexpert_v1/checkpoints/pt-imp-cxp
236 checkpoints available


## 2. How often is each pathology named in the training reports?

**What**
* f, the x-axis of both relationships: for each pathology, the share (%) of the 377,110 training image–text pairs whose report **asserts** the pathology.

    - relationship (1), section 7: x = log₁₀ f, y = zero-shot AUC
    - relationship (2), section 9: x = log₁₀ f, y = ΔAUC (few-shot minus zero-shot)
    - the section-10 figure plots both side by side with that x-axis

* **TODO**: check if we only treat asserted text. If report mentions "no ~~" it also is giving explicit hint to the model. It is not zeor shot.
* 📝 The unit is training *pairs*, not studies, because that's what the model saw: a study with 3 images shows its text 3 times.

**Primary definition (citable): the CheXpert labeler.** MIMIC ships the labeler's output for every
study (`mimic-cxr-2.0.0-chexpert.csv.gz`, path in `config/datasets.yaml`):
* mentions are found with a phrase list *"manually curated by multiple board-certified
  radiologists"* (Irvin et al. 2019, PDF p.2);
* each mention is classified positive / negative / uncertain, and a study is **positive (1)** if any
  mention is positive (Irvin PDF p.3);
* checked against a radiologist on 687 MIMIC reports (Johnson et al. 2019, PDF p.5).

✅ **`NO IMPRESSION` rows count as not exposed.** MIMIC labelled reports without an impression from
the findings section (Johnson PDF p.3), but our model only saw the text `NO IMPRESSION`
(57,000 of 377,110 rows).

**Every definition is computed; `frequency.primary` in `config/freq_v1.yaml` picks the one used:**

| column | definition | source |
|---|---|---|
| `labeler_pos_%` | labeler says positive (1) | Irvin PDF p.3, **default** |
| `labeler_pos_unc_%` | positive or uncertain (1 or −1) | CheXpert's U-Ones mapping, Irvin PDF p.4 |
| `labeler_any_%` | named at all (1, 0 or −1) | "mention extraction", Irvin PDF p.3 / Johnson PDF p.5 |
| `regex_%` | our hand-written synonyms + negation regex (`config/concept_synonyms.yaml`) | **none**: our own heuristic, robustness only |
| `udandarao_%` | **Udandarao et al.'s own method**, ported from their released code: spaCy noun lemmas; a row counts if its text contains every word of the class name (as prompted); no synonyms, no negation | Udandarao PDF p.3; code `bethgelab/frequency_determines_performance` @11ef009, spaCy 3.7.2 |

Reporting the result under each definition is a multiverse / specification-curve analysis
(Steegen et al. 2016; Simonsohn et al. 2020).

**`udandarao_%` details.** `utils/freq/udandarao_count.py` reproduces their indexing and search; on
3,000 real impressions it gives the same index and counts as their own functions. It runs in a
separate environment with their spaCy version (`frequency.spacy_python` in the YAML). Quirks of
their method, kept on purpose: a class-name word that never occurs as a noun is skipped, so
"Pleural Effusion" counts "effusion" alone; "Pleural Thickening" lemmatizes to *thicken* (a verb),
so it can count 0; and denials ("no pneumothorax") count as mentions.

**Status:**
* ✅ On the 12 CheXpert labels the definitions rank pathologies similarly (Spearman ρ with
  `labeler_pos_%`: 0.97 pos+unc, 0.85 any mention, 0.94 regex; computed 2026-10-05).
* ⚠ **The labeler covers only 7 of the 14 NIH labels** (Atelectasis, Cardiomegaly, Consolidation,
  Edema, Effusion, Pneumonia, Pneumothorax). With a `labeler_*` primary, NIH relationships use
  n = 7; the other 7 (Emphysema, Fibrosis, Hernia, Infiltration, Mass, Nodule, Pleural_Thickening)
  have f only from the regex. Mass and Nodule are merged into the labeler's "Lung Lesion" and
  can't be separated. A citable f for those 7 needs NegBio with UMLS (Peng et al. 2018, PDF p.2 and p.4).
* ⚠ Not yet run: MIMIC's NegBio label file (`mimic-cxr-2.0.0-negbio.csv.gz`) isn't on disk;
  download it from PhysioNet to add the "other labeler" variant.

In [11]:
IMP         = F["chexzero_data"]["impressions"]
CXP_ALL_CFG = CFG["cxp_labels_all"]
LABELS      = list(dict.fromkeys(NIH14 + CXP_ALL_CFG))
PROMPT_NAME = {l: LABEL_PROMPTS.get(l, l) for l in LABELS}      # class name as used in the prompts

regex, N_ROWS, cache = mentions.mention_table(IMP, CFG["synonyms"], OUT)                       # our hand regex
labeler = mentions.labeler_table(IMP, F["mimic"]["chexpert_labels"], LABELS)                  # CheXpert labeler
udan    = mentions.udandarao_table(IMP, sorted(set(PROMPT_NAME.values())),                     # Udandarao et al.
                                   os.path.join(OUT, "udandarao_counts.csv"), CFG["frequency"]["spacy_python"])
freq    = mentions.frequency(regex, labeler, primary=CFG["frequency"]["primary"],
                             udandarao={l: udan.loc[PROMPT_NAME[l], "udandarao_%"] for l in LABELS})

print(f"training rows (image-text pairs) : {N_ROWS:,}")
print(f"primary definition of f         : {CFG['frequency']['primary']}   (switch in config/freq_v1.yaml)")
print(f"labels with an f, NIH           : {freq.loc[NIH14, 'log_f'].notna().sum()} of {len(NIH14)}")
print(f"labels with an f, CheXpert      : {freq.loc[CXP_ALL_CFG, 'log_f'].notna().sum()} of {len(CXP_ALL_CFG)}\n")
VARIANTS = ["labeler_pos_%", "labeler_pos_unc_%", "labeler_any_%", "regex_%", "udandarao_%"]
freq.loc[LABELS, VARIANTS].sort_values("labeler_pos_%", ascending=False).round(2)

AttributeError: module 'utils.freq.mentions' has no attribute 'labeler_table'

### Why negation matters (hand-regex counts)

A pathology with many mentions but a high negated % means the model saw the *word* often but the
*finding* rarely. Radiologists write "no pneumothorax" to rule it out, so pneumothorax is the
extreme case. The table below uses the hand regex, which records raw mentions and denials; the
labeler variants show the same thing (compare `labeler_any_%` with `labeler_pos_%` above). Udandarao
et al. count caption matches with no negation handling (PDF p.3), which suits web captions but not
radiology reports.

In [ ]:
neg_view = regex.loc[NIH14, ["n_mentions", "negated_%", "affirmative_%"]].copy()     # hand regex counts
neg_view.insert(0, "mentions_%", 100 * neg_view.pop("n_mentions") / N_ROWS)
print("Pneumothorax: the word is common, the finding is not -- "
      f"{neg_view.loc['Pneumothorax', 'negated_%']:.0f}% of its mentions are denials (hand regex).")
neg_view.sort_values("negated_%", ascending=False).round(2)

## 3. Data splits

**What:** each dataset gets three roles that never share images: **selection** (choose the
checkpoint), **support** (the K few-shot images) and **test**.

| | selection | support (few-shot) | test |
|---|---|---|---|
| **CheXpert** | official val, 200 studies | 10,000 frontal images from CheXpert train | official test, 500 studies (radiologist consensus) |
| **ChestX-ray14** | 6,000 of the 20,000 promptdev images | the other 14,000 | official `test_list.txt`, 25,596 images |

**Status:**
* ✅ The NIH promptdev pool (20,000) lies entirely inside the official `train_val_list.txt`
  (86,524) and has 0 images and 0 patients in common with the test set (checked 2026-10-04).
* ✅ Selection and support are split from the pool by a fixed permutation (seed 0), so they are disjoint.
* ✅ Each h5 file stores only pixels. Its `*_paths.csv` records which image is in which row, and
  every label matrix is re-ordered through that file, never assumed (`utils/freq/data.py`).
* ⚠ The NIH selection/support split is by **image**, not patient, so the same patient can appear in
  both. That doesn't touch the test set, but it should be stated.
* ⚠ CheXpert support labels come from the CheXpert labeler (machine-extracted); test labels are
  radiologist consensus. See limitation 4.

In [ ]:
cz, cx = F["chexzero_data"], F["chexpert"]

# ---- h5 files built here (skipped when they exist) ----------------------------------
CXP_TEST_H5,  CXP_TEST_PATHS = os.path.join(EXP, "chexpert_test.h5"),    os.path.join(EXP, "chexpert_test_paths.csv")
CXP_SUPP_H5,  CXP_SUPP_PATHS = os.path.join(EXP, "chexpert_support.h5"), os.path.join(EXP, "chexpert_support_paths.csv")
data.build_chexpert_test(cx["test_images"], CXP_TEST_H5, CXP_TEST_PATHS)
data.build_chexpert_support(cx["train_csv"], CXP_SUPP_H5, CXP_SUPP_PATHS, E["seed"])

# ---- NIH: 20,000-image pool -> selection + support ------------------------------------
n_pool = len(pd.read_csv(cz["nih_pool_paths"]))
NIH_SEL_IDX, NIH_SUPP_IDX = data.split_pool(n_pool, E["n_sel"], E["seed"])
print(f"NIH pool {n_pool:,}  ->  selection {len(NIH_SEL_IDX):,}  +  support {len(NIH_SUPP_IDX):,}")

# ---- labels, each in its h5's row order ------------------------------------------------
y_cxp_test_full = data.chexpert_test_labels(cx["test_labels"], CXP_TEST_PATHS, CXP5)   # all columns, for section 12
y_cxp_supp_full = data.chexpert_support_labels(cx["train_csv"], CXP_SUPP_PATHS)
y_cxp_val       = cxp_val_groundtruth(cx["valid_csv"], cz["cxp_val_paths"], CXP5)
y_nih_test      = nih_groundtruth(F["nih"]["labels"], cz["nih_test_paths"], NIH14)
y_nih_pool      = nih_groundtruth(F["nih"]["labels"], cz["nih_pool_paths"], NIH14)

# ---- shape + leakage checks -------------------------------------------------------------
data.check_aligned([("CheXpert test", CXP_TEST_H5,         len(y_cxp_test_full)),
                    ("CheXpert supp", CXP_SUPP_H5,         len(y_cxp_supp_full)),
                    ("CheXpert val",  cz["cxp_val_h5"],    len(y_cxp_val)),
                    ("NIH test",      cz["nih_test_h5"],   len(y_nih_test)),
                    ("NIH pool",      cz["nih_pool_h5"],   len(y_nih_pool))])
print("\n  CheXpert test/support  shared paths %d, patients %d" % data.overlap(CXP_TEST_PATHS, CXP_SUPP_PATHS))
print("  NIH test/pool          shared images %d" % data.overlap(cz["nih_test_paths"], cz["nih_pool_paths"])[0])

## 4. Checkpoint selection: each dataset picks its own

**What:** every one of the 236 checkpoints is scored on the dataset's **selection** split (mean
zero-shot AUC), and the single best is kept. CheXpert uses its 5 competition labels; NIH uses all 14.

**Prompts:** CheXzero's frame, `"{label}"` versus `"no {label}"`.
* CheXpert: the label name as is.
* NIH: `LABEL_PROMPTS` from `CheXzero/eval_nih.py` (radiological phrasing, e.g. "Effusion" → "pleural effusion").

**Status:**
* ⚠ Check whether `LABEL_PROMPTS` were tuned on the same promptdev images
  (`search_prompts.py` defaults to them). If so, the 20,000 images chose both the prompts and the
  checkpoint. That's no leak into the test set, but it should be stated.
* ✅ The final `checkpoint.pt` has no step number in its name; it is ranked as step −1.
* Rankings are cached in `runs/freq_v1/results/ranking_*.csv`.

In [ ]:
CXP_PROMPT = {l: l for l in CXP5}
NIH_PROMPT = {l: LABEL_PROMPTS.get(l, l) for l in NIH14}
run = dict(batch=E["batch"], ctx=E["ctx"], device=DEVICE)

print("ranking on CheXpert val (200 studies) ...")
rank_cxp = clip_eval.rank_checkpoints(CKPT_PATHS, cz["cxp_val_h5"], CXP5, CXP_PROMPT, y_cxp_val,
                                      os.path.join(OUT, "ranking_chexpert.csv"), **run)
print("\nranking on held-out NIH split (6,000 images) ...")
rank_nih = clip_eval.rank_checkpoints(CKPT_PATHS, cz["nih_pool_h5"], NIH14, NIH_PROMPT, y_nih_pool[NIH_SEL_IDX],
                                      os.path.join(OUT, "ranking_nih.csv"), rows=NIH_SEL_IDX, **run)

BEST_CXP, BEST_NIH = rank_cxp.iloc[0], rank_nih.iloc[0]
SPE = CFG["steps_per_epoch"]
for name, b in [("CheXpert", BEST_CXP), ("NIH", BEST_NIH)]:
    print(f"{name:<9} best = step {b.step:>6}  (epoch {b.step/SPE:.2f})  selection AUC {b.mean_auc:.4f}")
print(f"same checkpoint chosen by both? {BEST_CXP.checkpoint == BEST_NIH.checkpoint}")

### Where does the best checkpoint fall, epoch by epoch?

The paper trains 4 epochs, a choice they tuned on their own validation runs. This table shows,
for each of our 8 epochs, the best checkpoint and the epoch's mean AUC, so we can see whether
training longer helped.

In [ ]:
for name, r in [("CheXpert", rank_cxp), ("NIH", rank_nih)]:
    print(f"\n{name}:")
    print(clip_eval.best_per_epoch(r, SPE, CFG["train"]["epochs"]).round(4).to_string())

## 5. Encode once, with each dataset's own checkpoint

**What:** image features don't depend on K, the seed or the prompt, so each image set is encoded
once with that dataset's chosen checkpoint and cached (`feat_*.npy`, keyed by checkpoint and file).
Everything afterwards is arithmetic on these vectors.

In [ ]:
enc = dict(cache_dir=OUT, batch=E["batch"], ctx=E["ctx"], device=DEVICE)
TASKS = {}
for name, ckpt, test_h5, supp_h5, y_test, y_supp, labels, pmap, supp_rows in [
    ("CheXpert", BEST_CXP.checkpoint, CXP_TEST_H5, CXP_SUPP_H5,
     y_cxp_test_full[CXP5].to_numpy(float), y_cxp_supp_full[CXP5].to_numpy(float), CXP5, CXP_PROMPT, None),
    ("NIH", BEST_NIH.checkpoint, cz["nih_test_h5"], cz["nih_pool_h5"],
     y_nih_test, y_nih_pool[NIH_SUPP_IDX], NIH14, NIH_PROMPT, NIH_SUPP_IDX)]:
    print(f"{name}  (checkpoint {os.path.basename(ckpt)})")
    TASKS[name] = clip_eval.make_task(ckpt, labels, pmap, y_test, y_supp,
                                      F_test=clip_eval.encode_images(test_h5, ckpt, tag=f"{name}_test", **enc),
                                      F_supp=clip_eval.encode_images(supp_h5, ckpt, tag=f"{name}_supp", rows=supp_rows, **enc),
                                      ctx=E["ctx"], device=DEVICE)
print("\nall features cached")

## 6. Zero-shot

$$p(\text{label}) = \frac{e^{\,v\cdot t_{pos}}}{e^{\,v\cdot t_{pos}} + e^{\,v\cdot t_{neg}}} = \sigma(v\cdot t_{pos} - v\cdot t_{neg})$$

v is the image feature; t_pos and t_neg are the "{label}" and "no {label}" text features. The code
uses the sigmoid form, which is identical and numerically stable.

**Reference:** 0.878, the CheXzero best single model (Table 5, PDF p.6), on the 5 CheXpert
competition labels. ⚠ That table is captioned as **validation** results (see the top cell).

The second cell gives a bootstrap 95% CI over test images (1,000 resamples).

In [ ]:
REF = CFG["reference"]["chexzero_single"]
for d in TASKS.values():
    clip_eval.zero_shot(d)

for name, d in TASKS.items():
    print(f"\n=== {name} zero-shot, {len(d['labels'])} labels ===")
    print(pd.Series(d["zs_auc"]).round(4).to_string())
    print(f"MEAN  {d['ZS']:.4f}")
print(f"\nCheXpert  ours {TASKS['CheXpert']['ZS']:.4f}   CheXzero single model {REF}   "
      f"diff {TASKS['CheXpert']['ZS'] - REF:+.4f}")

In [ ]:
for name, d in TASKS.items():
    m, lo, hi = laws.boot_mean_auc(d["y_test"], d["zs_prob"], d["labels"], n_boot=E["n_boot"])
    note = f"   {REF} inside CI: {'yes' if lo <= REF <= hi else 'no'}" if name == "CheXpert" else ""
    print(f"{name:<9} {m:.4f}  95% CI [{lo:.4f}, {hi:.4f}]{note}")

## 7. Relationship (1), "Law 1": frequency vs zero-shot AUC

**What:** regress NIH test AUC on log₁₀ f, one point per pathology: n = 7 with the default
(labeler) f, 14 with the regex f (section 2). Pathologies without an f drop out.

**Source:** Udandarao et al. 2024. Our version differs from theirs in three ways; say so in the paper:
* they fit thousands of concepts, averaged within log-frequency bins for plotting (PDF p.6); we fit 7–14 pathologies, one point each;
* they report Pearson ρ with a two-tailed t-test (PDF p.6); we add **Spearman**, because 7–14 points is few;
* their main f is image ∩ text matches; ours is text only. They show text-only f gives the same
  trend (Appendix D, PDF p.25).

**Checks in the cell:**
* a rare/common split at the median f, tested non-parametrically (Mann-Whitney);
* the obvious confound: rare pathologies have fewer test positives, so their AUC is noisier.
  The partial correlation removes log(test positives) from both variables.

In [ ]:
L1 = laws.law1(freq, TASKS["NIH"]["zs_auc"], y_nih_test.sum(0).astype(int), NIH14)

print(f"LAW 1   AUC = {L1['slope']:+.4f} * log10(f) + {L1['intercept']:.4f}      n = {len(L1['table'])}")
print(f"        pearson  r = {L1['r']:+.3f}  p = {L1['p']:.4f}")
print(f"        spearman rho = {L1['rho']:+.3f}  p = {L1['ps']:.4f}\n")
for g in ["rare", "common"]:
    t = L1[g]
    print(f"{g.upper():<6} n={len(t)}  mean f {t['f_%'].mean():5.2f}%  mean AUC {t.auc.mean():.4f}")
print(f"gap = {L1['common'].auc.mean() - L1['rare'].auc.mean():+.4f}   Mann-Whitney p = {L1['split_p']:.4f}\n")
print(f"CONFOUND  test positives vs AUC:  r = {L1['confound_r']:+.3f}  p = {L1['confound_p']:.4f}")
print(f"          partial r controlling for it: {L1['partial_r']:+.3f}  p = {L1['partial_p']:.4f}   "
      f"-> {'survives' if L1['partial_p'] < 0.05 else 'DOES NOT survive'}")

## 8. Few-shot: SS-Text

$$w_{pos} = \sum_{i \in \text{pos}} v_i + t_{pos}, \qquad w_{neg} = \sum_{i \in \text{neg}} v_i + t_{neg}$$

then score exactly as in zero-shot. Nothing in the encoder is trained.

**Source:** Silva-Rodríguez et al., *Full Conformal Adaptation of Medical Vision-Language Models*,
§4.3, Eq. 10 (PDF p.7). With their choice λ = 1/(Nτ), Eq. 10 reduces to the sum above.

**Notes:**
* It's a **sum, not a mean**: with K images per class the text vector is about 1/(K+1) of the
  classifier (about 6% at K = 16). Adaptation largely *replaces* the prompt rather than refining it.
* ⚠ Their released code (`FCA/solvers/sstext.py`) sets λ = 1/N and divides the image sum by τ, i.e.
  w = Σv/τ + t, so images weigh 1/τ times more than in the paper (about 100× for CLIP's usual τ ≈ 0.01). We follow the **paper**. Run the
  code's version as a check, since "adaptation hurts common pathologies" depends on this weighting.
* CheXpert support pools: label 1 = positive; 0 or blank = negative (the official convention);
  −1 (uncertain) is excluded. NIH has no blanks or −1.
* Each K is repeated with 20 random draws of the support images.

**The K = 0 check (next cell):** ⚠ with K = 0 the code takes the "no support images" branch, the
same code as zero-shot. So it only checks that the text path is unchanged; it does **not** test
the support-sum path. A real check would force the sum with an empty set.

In [ ]:
for name, d in TASKS.items():
    diff = fewshot.k0_matches_zero_shot(d)
    print(f"{name:<9} K=0 reproduces zero-shot: max abs diff {diff:.2e}   {'OK' if diff < 1e-9 else 'MISMATCH - STOP'}")

In [ ]:
fs = fewshot.sweep(TASKS, E["k_sweep"], E["seeds"], os.path.join(OUT, "fewshot_sweep.csv"))
summary = fewshot.summarize(fs)
for name, d in TASKS.items():
    s = summary[summary.dataset == name].set_index("K")
    s["vs_zero_shot"] = s["mean"] - d["ZS"]
    print(f"\n=== {name} ({len(d['labels'])} labels)   zero-shot {d['ZS']:.4f} ===")
    print(s[["mean", "std", "vs_zero_shot"]].round(4).to_string())

## 9. Relationship (2), "Law 2": frequency vs the gain from few-shot

**What:** the same x-axis as (1); the y-axis is now ΔAUC = few-shot AUC − zero-shot AUC, per
pathology, averaged over the 20 seeds. Fitted for every dataset and K. The hypothesis is a
**negative** slope: the more often a pathology is named, the more adaptation hurts.

**Status: ⚠ open issues to resolve before claiming (2):**
* **Regression to the mean.** Δ contains zero-shot AUC, and zero-shot AUC already rises with f
  (relationship 1). Pathologies near AUC 0.5 have more room to improve, so a negative slope can
  appear even if f plays no role of its own. Needed: the partial correlation of Δ with log f,
  controlling for zero-shot AUC, **and** few-shot AUC itself regressed on log f. *Not yet in the notebook.*
* **Choice of K.** The headline uses K = 64 (`k_ref` in the YAML); an earlier report used K = 16.
  Fix the choice in advance, or report all K.
* **Multiple tests.** The ★ marks min(Pearson p, Spearman p) < 0.05 across 7 values of K, with no
  correction. Treat it as descriptive.

In [ ]:
L2 = laws.law2(fs, TASKS, E["k_sweep"], freq)
fits = pd.DataFrame([{"dataset": n, "K": K, **{k: v for k, v in f.items() if k != "table"}}
                     for (n, K), f in L2.items()])
fits["sig"] = np.where(fits[["p", "ps"]].min(axis=1) < 0.05, "*", "")
print("* = significant at 0.05 on Pearson or Spearman")
fits[["dataset", "K", "r", "p", "rho", "ps", "slope", "sig"]].round(4)

In [ ]:
K_REF = E["k_ref"]
st = L2[("NIH", K_REF)]
t = st["table"].sort_values("delta", ascending=False)
print(f"ChestX-ray14, K={K_REF}: {int((t.delta > 0).sum())} of {len(t)} improved")
print(f"LAW 2   dAUC = {st['slope']:+.4f} * log10(f) + {st['intercept']:.4f}")
print(f"        pearson r = {st['r']:+.3f} (p={st['p']:.4f})   spearman rho = {st['rho']:+.3f} (p={st['ps']:.4f})")
t[["f_%", "zero_shot", "few_shot", "delta"]].round(4)

### Does frequency predict the *sign*?

A regression on 14 points can be driven by one outlier. Non-parametric check: split at the median f
and compare the two groups' Δ directly (one-sided Mann-Whitney: rare gains more than common).

In [ ]:
pd.DataFrame([{"dataset": n, "K": K, **laws.sign_test(L2[(n, K)]["table"])}
              for n in TASKS for K in E["k_report"]]).round(4)

## 10. The sign flip and the crossing point

**What:** if (1) rises with f and (2) falls, the line (2) crosses Δ = 0 at some f. Below it adaptation
helps; above it, keep zero-shot. That would be a rule that needs only the training reports.

**Status:** ⚠ **The crossing point is fitted on the NIH test set and then scored on the same test set**,
so "rule agrees with the outcome on X/14" is in-sample and can't support the rule. A fair version
fits the crossing on a separate split (e.g. the NIH selection images) or on another dataset.
The CheXpert rows in section 11 are the only out-of-sample check so far.

In [ ]:
cross = laws.crossing(st)
print(f"LAW 1   zero-shot AUC   = {L1['slope']:+.4f} * log10(f) + {L1['intercept']:.4f}")
print(f"LAW 2   few-shot delta  = {st['slope']:+.4f} * log10(f) + {st['intercept']:.4f}")
print(f"\nLaw 2 crosses zero at f = {cross:.2f}% (f, primary definition)")
side = laws.rule_vs_outcome(st["table"], cross)
print(f"rule agrees with the outcome on {(side.predicted == side.actual).sum()}/{len(side)} pathologies")
side[["f_%", "delta", "predicted", "actual"]].round(3)

In [ ]:
plot.two_laws(L1, st, cross, K_REF, out_png=os.path.join(OUT, "two_laws.png"))

## 11. Selective adaptation: what the rule is worth

Mean AUC under four policies:

| policy | what it does |
|---|---|
| **zero_shot** | adapt nothing |
| **uniform** | adapt every pathology, as SS-Text does |
| **rule** | adapt only where f < crossing point (training reports only) |
| **oracle** | adapt only where it actually helped; uses **test labels**, so an upper bound, not achievable |

⚠ For NIH, *rule* is in-sample (see section 10); only the CheXpert rows test it fairly.

In [ ]:
pol = laws.policies(L2, TASKS, E["k_report"], cross)
pol.to_csv(os.path.join(OUT, "policies.csv"), index=False)
pol.round(4)

## 12. Falsification test: CheXpert's own rare pathologies

**Why:** the 5 CheXpert competition labels are all common in MIMIC, and in the earlier run
(`results/chexpert_v1/`) all of them lost from adaptation. A reviewer could say CheXpert is simply a
dataset where few-shot fails. The hypothesis predicts that CheXpert's **rare** findings should
**gain**: same dataset, same model, opposite sign.

**What:** all 12 CheXpert pathologies (`cxp_labels_all` in the YAML; not Support Devices or
No Finding), zero-shot vs few-shot at K = 16 and 64, then (2) fitted on CheXpert alone.

**Caution:** the CheXpert test set has 500 studies, so the rare labels have very few positives:

| label | test positives |
|---|---|
| Pleural Other | 4 |
| Fracture | 5 |
| Lung Lesion | 8 |
| Pneumothorax | 9 |
| Pneumonia | 11 |

With 4–11 positives an AUC's 95% CI is roughly ±0.1–0.15, as large as the effects being measured.
Read the **direction**, not the magnitude. ⚠ Per-label CIs are not computed yet.
(The output file is still named `chexpert_13.csv` for continuity, though it holds 12 labels.)

In [ ]:
CXP_ALL = [l for l in CFG["cxp_labels_all"] if l in freq.index and freq.loc[l, "f_%"] > 0]
d = TASKS["CheXpert"]
d_all = clip_eval.zero_shot(clip_eval.make_task(
    d["ckpt"], CXP_ALL, {l: l for l in CXP_ALL},
    y_cxp_test_full[CXP_ALL].to_numpy(float), y_cxp_supp_full[CXP_ALL].to_numpy(float),
    d["F_test"], d["F_supp"], ctx=E["ctx"], device=DEVICE))
few = fewshot.per_label(d_all, E["k_report"], E["seeds"])

cxp_all = pd.DataFrame([{"K": K, "label": l, "few_shot": few[K][l], "zero_shot": d_all["zs_auc"][l],
                         "f_%": freq.loc[l, "f_%"],
                         "n_pos": int(d_all["y_test"][:, i].sum())}
                        for K in E["k_report"] for i, l in enumerate(CXP_ALL)])
cxp_all["delta"] = cxp_all.few_shot - cxp_all.zero_shot
cxp_all.to_csv(os.path.join(OUT, "chexpert_13.csv"), index=False)

t64 = cxp_all[cxp_all.K == K_REF].sort_values("f_%")
L2_cxp = laws.fit(np.log10(t64["f_%"]), t64.delta)
print(f"CheXpert, {len(CXP_ALL)} pathologies, K={K_REF}: {int((t64.delta > 0).sum())} of {len(t64)} improved")
print(f"Law 2 on CheXpert alone:  pearson r = {L2_cxp['r']:+.3f} (p={L2_cxp['p']:.4f})   "
      f"spearman rho = {L2_cxp['rho']:+.3f} (p={L2_cxp['ps']:.4f})")
print("-> same sign as ChestX-ray14?  " + ("YES" if L2_cxp["r"] < 0 else "NO - the hypothesis fails here"))
t64.set_index("label")[["f_%", "n_pos", "zero_shot", "few_shot", "delta"]].round(4)

## 13. Everything in one table

The chain of evidence, end to end. ⚠ The **claim** column is the *hypothesis* as written in
advance, not a verdict. Read the **evidence** column (computed) to judge whether each claim holds.

In [ ]:
final = pd.DataFrame([
    {"step": "1. Reproduction", "claim": "model matches the published single-model figure",
     "evidence": f"CheXpert test {TASKS['CheXpert']['ZS']:.4f} vs CheXzero {REF}"},
    {"step": "2. Law 1 (replication)", "claim": "rarely-named pathologies score worse zero-shot",
     "evidence": f"slope {L1['slope']:+.4f}/decade, r={L1['r']:+.2f}, p={L1['p']:.3f}; "
                 f"partial r={L1['partial_r']:+.2f}, p={L1['partial_p']:.3f} controlling for test prevalence"},
    {"step": "3. Law 2 (new)", "claim": "rarely-named pathologies benefit MORE from adaptation",
     "evidence": f"slope {st['slope']:+.4f}/decade, rho={st['rho']:+.2f}, p={st['ps']:.4f} (NIH, K={K_REF})"},
    {"step": "4. Sign flip", "claim": "one measurement, opposite signs; the lines cross",
     "evidence": f"Law 1 {L1['slope']:+.3f} vs Law 2 {st['slope']:+.3f}; crossing at f={cross:.1f}%"},
    {"step": "5. Decision rule", "claim": "a label-free rule beats adapting everything",
     "evidence": "see section 11"},
    {"step": "6. Falsification", "claim": "holds on CheXpert's own rare pathologies",
     "evidence": f"Law 2 on CheXpert: rho={L2_cxp['rho']:+.2f}, p={L2_cxp['ps']:.4f}"},
])
final.to_csv(os.path.join(OUT, "evidence_chain.csv"), index=False)
pd.set_option("display.max_colwidth", 95)
final.style.hide(axis="index")

## 14. Limitations

1. **n = 7 pathologies** (NIH, labeler f; 14 with the regex f) and 12 (CheXpert). Small for a regression; Spearman and a
   non-parametric split are reported throughout. PadChest (many findings with > 50 positives)
   would fix this.
2. **The frequency definition** (section 2): the primary f now comes from the CheXpert labeler, with
   `NO IMPRESSION` rows counted as not exposed. It covers only 7 of the 14 NIH labels; the other 7
   have only our hand regex, which has no published source.
3. **Regression to the mean** may produce relationship (2) from relationship (1) alone (section 9).
   The control is not implemented yet.
4. **CheXpert support labels are machine-extracted**; test labels are radiologist consensus. Label
   noise is a competing explanation for CheXpert's weaker few-shot results and is **not**
   separated from frequency by this design. NIH shows the same pattern with its own labels, which
   argues for frequency but doesn't prove it.
5. **The crossing point is in-sample** (section 10).
6. **SS-Text weighting:** we follow the paper's Eq. 10 (a sum), not the authors' code (Σv/τ). The
   result may depend on this (section 8).
7. **Two checkpoints:** each dataset selects its own, so cross-dataset magnitudes compare
   different models. Within-dataset results are primary.
8. **No ensemble:** the reference is 0.878 (single model), which CheXzero reports on the validation set.
9. **Correlation, not causation.** A causal test would remove one pathology's mentions from the
   training text, retrain, and check that its few-shot gain rises: one training run per pathology.

In [ ]:
print("saved to", OUT)
for f in sorted(os.listdir(OUT)):
    print(f"  {f:<42}{os.path.getsize(os.path.join(OUT, f))/1e6:>8.1f} MB")